In [2]:
from pathlib import Path
while not (Path.cwd() / '.git').exists():
    %cd ..

/home/matthew/study/grab-voc-triage


In [11]:
import transformers
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset
import pandas as pd
import config
import torch
import dotenv
import os
from torch import nn
dotenv.load_dotenv(override = True)

True

In [4]:
tokenizer = AutoTokenizer.from_pretrained("models/indobert-base-p1")

In [7]:
class ReviewDataset(Dataset):
    def __init__(self, parquet_path : str, pretrained_model : str, max_len : int):
        df = pd.read_parquet(parquet_path)
        self.texts = df['content'].astype('str').to_list()
        self.labels = df[config.CATEGORIES].values.astype('float32')
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_model)
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            text = self.texts[idx],
            padding = 'max_length',
            truncation = True,
            max_length = self.max_len,
            return_tensors = 'pt'
        )
        return {
            'input_ids' : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'labels' : torch.tensor(self.labels[idx])
        }
dataset = ReviewDataset('data/processed/test.parquet', 'models/indobert-base-p1', 128)
dataset[0]['labels']


tensor([0., 0., 0.])

In [8]:
indobert_model = AutoModel.from_pretrained("models/indobert-base-p1", token = os.getenv('HF_TOKEN'))

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 4910.99it/s]


In [9]:
class ReviewClassifier(nn.Module):
    def __init__(self, 

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=